# 🌍 Geopolitical State Classification using Neural Networks
**Predicts the relationship between two countries into 5 categories:**
- 0 → Peace
- 1 → Strategic Competition
- 2 → Cold War
- 3 → Proxy War
- 4 → Direct War

---
## 📋 Roadmap
| Step | Task |
|------|------|
| 1 | Data Loading & Exploration (EDA) |
| 2 | Feature Engineering |
| 3 | Data Splitting (70 / 15 / 15) |
| 4 | Standardization (3×) |
| 5 | Build Neural Network Model |
| 6 | Train the Model |
| 7 | Evaluate & Visualize |
| 8 | Hyperparameter Tuning |

---
## ✅ STEP 0 — Install & Import Libraries
Run this cell first to make sure every library is available.

In [ ]:
# ── Install (only needed once, skip if already installed) ──────────────────
# !pip install tensorflow scikit-learn pandas numpy matplotlib seaborn keras-tuner

# ── Core libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

# ── TensorFlow / Keras ────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt

# ── Reproducibility seed ─────────────────────────────────────────────────
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported successfully!")
print(f"   TensorFlow version : {tf.__version__}")

---
## 📂 STEP 1 — Data Loading & Exploratory Data Analysis (EDA)

In [ ]:
# ── 1-A  Load the dataset ─────────────────────────────────────────────────
df = pd.read_csv('geopolitical_dataset.csv')   # ← update path if needed

print("📌 Shape of dataset:", df.shape)
print("\n📌 First 5 rows:")
df.head()

In [ ]:
# ── 1-B  Basic info ───────────────────────────────────────────────────────
print("📌 Column data types & non-null counts:")
df.info()

In [ ]:
# ── 1-C  Statistical summary ──────────────────────────────────────────────
print("📌 Statistical summary of all features:")
df.describe().round(3)

In [ ]:
# ── 1-D  Check for missing values ─────────────────────────────────────────
missing = df.isnull().sum()
print("📌 Missing values per column:")
print(missing)
print("\n✅ Total missing cells:", missing.sum())

In [ ]:
# ── 1-E  Target class distribution ────────────────────────────────────────
label_names = {0: 'Peace', 1: 'Strategic Competition', 2: 'Cold War',
               3: 'Proxy War', 4: 'Direct War'}

counts = df['target'].value_counts().sort_index()
print("📌 Class distribution:")
for k, v in counts.items():
    print(f"   Class {k} ({label_names[k]}): {v} samples  ({v/len(df)*100:.1f}%)")

# Bar chart
plt.figure(figsize=(8, 4))
plt.bar([label_names[i] for i in counts.index], counts.values,
        color=['green','steelblue','orange','red','darkred'])
plt.title('Target Class Distribution')
plt.xlabel('Geopolitical State')
plt.ylabel('Number of Samples')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print("\n⚠️  NOTE: Class imbalance detected — class weights will be applied during training.")

In [ ]:
# ── 1-F  Correlation heatmap ──────────────────────────────────────────────
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── 1-G  Feature distributions by class ──────────────────────────────────
key_features = ['military_power_ratio', 'gdp_ratio', 'news_sentiment',
                'ideology_distance', 'nationalism_index']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, feat in zip(axes, key_features):
    for cls in sorted(df['target'].unique()):
        ax.hist(df[df['target'] == cls][feat], bins=20, alpha=0.5,
                label=label_names[cls])
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=6)
plt.suptitle('Key Feature Distributions by Class', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 🛠️ STEP 2 — Feature Engineering
We create new meaningful features from combinations of existing ones.

In [ ]:
# Work on a copy so the original data stays safe
data = df.copy()

# ── NEW FEATURE 1: Military-Economic Tension Index ────────────────────────
# Combines military power imbalance + sanctions pressure
data['mil_econ_tension'] = data['military_power_ratio'] * data['sanctions_intensity']

# ── NEW FEATURE 2: Nuclear Threat Score ───────────────────────────────────
# Nuclear-capable countries with many warheads = higher threat
data['nuclear_threat'] = data['nuclear_capability'] * data['nuclear_warheads']

# ── NEW FEATURE 3: Ideological-Nationalism Friction ──────────────────────
# High ideology gap + high nationalism = more friction
data['ideol_nation_friction'] = data['ideology_distance'] * data['nationalism_index']

# ── NEW FEATURE 4: Conflict Proximity Score ───────────────────────────────
# Countries with historical conflict AND short border distance = higher risk
# Add 1 to border_distance to avoid division by zero
data['conflict_proximity'] = data['historical_conflict'] / (data['border_distance'] + 1)

# ── NEW FEATURE 5: Diplomatic Stability Index ─────────────────────────────
# High trade dependency + high political stability + positive sentiment = stable
data['diplomatic_stability'] = (data['trade_dependency'] *
                                 data['political_stability'] *
                                 (data['news_sentiment'] + 1))  # shift to positive range

# ── NEW FEATURE 6: Aggression Multiplier ──────────────────────────────────
# Leader aggression scaled by troop movement
data['aggression_multiplier'] = data['leader_aggression'] * data['troop_movement']

print("✅ Feature Engineering complete!")
print(f"   Original features : 14")
print(f"   New features added : 6")
print(f"   Total features     : {data.shape[1] - 1}  (excluding target)")

# Show a few new features
data[['mil_econ_tension','nuclear_threat','ideol_nation_friction',
      'conflict_proximity','diplomatic_stability','aggression_multiplier']].head(3)

---
## ✂️ STEP 3 — Data Splitting

**Strategy (as defined in the project):**
```
Full Dataset  (100%)
     ├── Train Set      → 70%  (used to train the model)
     └── Temp Set       → 30%
           ├── Validation Set → 15%  (tune & monitor training)
           └── Test Set       → 15%  (final unseen evaluation)
```

In [ ]:
# ── Separate features (X) and target (y) ─────────────────────────────────
X = data.drop(columns=['target'])
y = data['target']

print("📌 Feature matrix shape :", X.shape)
print("📌 Target vector shape  :", y.shape)
print("📌 Feature names:")
print(list(X.columns))

In [ ]:
# ── SPLIT 1: 70% Train  |  30% Temp ───────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,       # 30% goes to temp
    random_state=42,
    stratify=y            # keeps class proportions balanced in each split
)

# ── SPLIT 2: 50% of Temp = Validation  |  50% of Temp = Test ─────────────
# (50% of 30% = 15% each of the whole dataset)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,       # 50-50 split of the temp 30%
    random_state=42,
    stratify=y_temp
)

# ── Summary ───────────────────────────────────────────────────────────────
total = len(X)
print("📊 Data Split Summary:")
print(f"   ✅ Train      : {len(X_train):>4} samples  ({len(X_train)/total*100:.1f}%)")
print(f"   ✅ Validation : {len(X_val):>4} samples  ({len(X_val)/total*100:.1f}%)")
print(f"   ✅ Test       : {len(X_test):>4} samples  ({len(X_test)/total*100:.1f}%)")
print(f"   ─────────────────────")
print(f"      Total     : {total} samples")

In [ ]:
# ── Verify class balance is preserved in each split ───────────────────────
splits = {'Train': y_train, 'Validation': y_val, 'Test': y_test}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, y_split) in zip(axes, splits.items()):
    counts = y_split.value_counts().sort_index()
    ax.bar([label_names[i] for i in counts.index], counts.values,
           color=['green','steelblue','orange','red','darkred'])
    ax.set_title(f'{name} Set Class Distribution')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

print("✅ Stratified split ensures class proportions are preserved!")

---
## 📏 STEP 4 — Standardization (3 Times)

**Rule:** Fit the scaler ONLY on training data, then transform all 3 sets.

> This prevents **data leakage** — the model must not "see" validation or test statistics during training.

```
StandardScaler.fit(X_train)          ← learns mean & std from train only
    │
    ├── transform(X_train)  → X_train_scaled  (1st standardization)
    ├── transform(X_val)    → X_val_scaled    (2nd standardization)
    └── transform(X_test)   → X_test_scaled   (3rd standardization)
```

In [ ]:
# ── Create the scaler ─────────────────────────────────────────────────────
scaler = StandardScaler()

# ── Fit ONLY on training data ─────────────────────────────────────────────
scaler.fit(X_train)

# ── 1st Standardization → Transform TRAIN ─────────────────────────────────
X_train_scaled = scaler.transform(X_train)
print("✅ 1st Standardization — Train set transformed")
print(f"   Mean (first 3 features): {X_train_scaled.mean(axis=0)[:3].round(4)}")
print(f"   Std  (first 3 features): {X_train_scaled.std(axis=0)[:3].round(4)}")

# ── 2nd Standardization → Transform VALIDATION ────────────────────────────
X_val_scaled = scaler.transform(X_val)
print("\n✅ 2nd Standardization — Validation set transformed")

# ── 3rd Standardization → Transform TEST ──────────────────────────────────
X_test_scaled = scaler.transform(X_test)
print("✅ 3rd Standardization — Test set transformed")

print("\n📌 All 3 standardizations complete!")
print(f"   X_train_scaled shape : {X_train_scaled.shape}")
print(f"   X_val_scaled   shape : {X_val_scaled.shape}")
print(f"   X_test_scaled  shape : {X_test_scaled.shape}")

In [ ]:
# ── Convert labels to numpy arrays ────────────────────────────────────────
y_train_np = y_train.values
y_val_np   = y_val.values
y_test_np  = y_test.values

NUM_CLASSES  = 5
NUM_FEATURES = X_train_scaled.shape[1]

print(f"📌 Number of features : {NUM_FEATURES}")
print(f"📌 Number of classes  : {NUM_CLASSES}")
print(f"   Labels             : {sorted(np.unique(y_train_np))}")

---
## 🧠 STEP 5 — Build the Neural Network Model

**Architecture:**
```
Input (20 features)
    → Dense(128, ReLU) + BatchNorm + Dropout(0.3)
    → Dense(64,  ReLU) + BatchNorm + Dropout(0.3)
    → Dense(32,  ReLU) + BatchNorm + Dropout(0.2)
    → Output Dense(5, Softmax)   ← 5 classes
```

In [ ]:
def build_model(input_dim, num_classes):
    """
    Builds a Multi-Layer Neural Network for multi-class classification.
    
    Architecture:
        Input → Dense(128) → Dense(64) → Dense(32) → Output(5)
        Each hidden layer uses: ReLU activation, BatchNormalization, Dropout
    """
    model = keras.Sequential([

        # ── INPUT LAYER ────────────────────────────────────────────────────
        keras.Input(shape=(input_dim,), name='Input'),

        # ── HIDDEN LAYER 1 (Widest) ────────────────────────────────────────
        layers.Dense(128, name='Dense_1'),            # 128 neurons
        layers.BatchNormalization(name='BN_1'),       # normalise activations
        layers.Activation('relu', name='ReLU_1'),     # ReLU: f(x) = max(0,x)
        layers.Dropout(0.3, name='Dropout_1'),        # randomly zero 30% neurons

        # ── HIDDEN LAYER 2 ─────────────────────────────────────────────────
        layers.Dense(64, name='Dense_2'),
        layers.BatchNormalization(name='BN_2'),
        layers.Activation('relu', name='ReLU_2'),
        layers.Dropout(0.3, name='Dropout_2'),

        # ── HIDDEN LAYER 3 (Narrowest) ─────────────────────────────────────
        layers.Dense(32, name='Dense_3'),
        layers.BatchNormalization(name='BN_3'),
        layers.Activation('relu', name='ReLU_3'),
        layers.Dropout(0.2, name='Dropout_3'),

        # ── OUTPUT LAYER ───────────────────────────────────────────────────
        # Softmax gives probabilities for each of the 5 classes (sum = 1)
        layers.Dense(num_classes, activation='softmax', name='Output')
    ], name='Geopolitical_Classifier')

    return model


# ── Build the model ───────────────────────────────────────────────────────
model = build_model(input_dim=NUM_FEATURES, num_classes=NUM_CLASSES)

# ── Print a readable summary ──────────────────────────────────────────────
model.summary()

In [ ]:
# ── Compute class weights to handle imbalance ─────────────────────────────
# Since classes like 'Direct War' (class 4) are very rare,
# we give them more weight so the model pays more attention to them.

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_np),
    y=y_train_np
)
class_weight_dict = dict(enumerate(class_weights_array))

print("📌 Class weights (higher = rarer class):")
for k, v in class_weight_dict.items():
    print(f"   Class {k} ({label_names[k]}): {v:.4f}")

In [ ]:
# ── Compile the model ─────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # use this when labels are integers
    metrics=['accuracy']
)

print("✅ Model compiled!")
print("   Optimizer  : Adam (lr=0.001)")
print("   Loss       : sparse_categorical_crossentropy")
print("   Metric     : Accuracy")

---
## 🏋️ STEP 6 — Train the Model

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss',    # watch validation loss
    patience=15,           # stop if no improvement for 15 epochs
    restore_best_weights=True,  # revert to best epoch automatically
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,            # halve the learning rate
    patience=7,            # wait 7 epochs before reducing
    min_lr=1e-6,
    verbose=1
)

# ── Train ─────────────────────────────────────────────────────────────────
print("🚀 Starting training...\n")
history = model.fit(
    X_train_scaled, y_train_np,         # training data
    validation_data=(X_val_scaled, y_val_np),  # validation data
    epochs=100,                          # max epochs (early stop will kick in)
    batch_size=32,                       # 32 samples per gradient update
    class_weight=class_weight_dict,      # handle class imbalance
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

---
## 📊 STEP 7 — Evaluate & Visualize Results

In [ ]:
# ── 7-A  Training History Plot ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss',      color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(history.history['accuracy'],     label='Train Accuracy',      color='blue')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training History', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7-B  Evaluate on all 3 sets ───────────────────────────────────────────
train_loss, train_acc = model.evaluate(X_train_scaled, y_train_np, verbose=0)
val_loss,   val_acc   = model.evaluate(X_val_scaled,   y_val_np,   verbose=0)
test_loss,  test_acc  = model.evaluate(X_test_scaled,  y_test_np,  verbose=0)

print("📊 Model Performance:")
print(f"{'Set':<15} {'Loss':>10} {'Accuracy':>10}")
print("-" * 38)
print(f"{'Train':<15} {train_loss:>10.4f} {train_acc*100:>9.2f}%")
print(f"{'Validation':<15} {val_loss:>10.4f} {val_acc*100:>9.2f}%")
print(f"{'Test':<15} {test_loss:>10.4f} {test_acc*100:>9.2f}%")

In [ ]:
# ── 7-C  Confusion Matrix ────────────────────────────────────────────────
y_pred_probs = model.predict(X_test_scaled)
y_pred       = np.argmax(y_pred_probs, axis=1)   # pick class with highest probability

cm = confusion_matrix(y_test_np, y_pred)
class_names = [label_names[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', ax=plt.gca(), colorbar=False)
plt.title('Confusion Matrix — Test Set')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7-D  Classification Report ────────────────────────────────────────────
print("📋 Classification Report (Test Set):")
print(classification_report(
    y_test_np, y_pred,
    target_names=class_names,
    zero_division=0
))

In [ ]:
# ── 7-E  Per-class confidence analysis ────────────────────────────────────
print("📊 Average Prediction Confidence per True Class (Test Set):")
for cls in range(NUM_CLASSES):
    mask = (y_test_np == cls)
    if mask.sum() > 0:
        avg_conf = y_pred_probs[mask, cls].mean()
        print(f"   Class {cls} ({label_names[cls]}): {avg_conf*100:.1f}%  (n={mask.sum()})")
    else:
        print(f"   Class {cls} ({label_names[cls]}): No samples in test set")

---
## ⚙️ STEP 8 — Hyperparameter Tuning with Keras Tuner
We search for the best: number of neurons, dropout rate, and learning rate.

> 💡 This may take a few minutes. Keras Tuner tries many combinations automatically.

In [ ]:
# ── Define a model-building function for the tuner ────────────────────────
def build_tunable_model(hp):
    """
    hp = HyperParameters object. The tuner will call this function
    many times with different hp values to find the best combination.
    """
    model = keras.Sequential(name='Tunable_Geopolitical_Classifier')
    model.add(keras.Input(shape=(NUM_FEATURES,)))

    # Search over 1, 2, or 3 hidden layers
    for i in range(hp.Int('num_layers', min_value=1, max_value=3)):
        units = hp.Choice(f'units_{i}', values=[32, 64, 128, 256])
        dropout = hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)

        model.add(layers.Dense(units))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation('relu'))
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # Search over learning rates
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 5e-4, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# ── Set up the Hyperband tuner ─────────────────────────────────────────────
tuner = kt.Hyperband(
    build_tunable_model,
    objective='val_accuracy',      # maximise validation accuracy
    max_epochs=30,
    factor=3,
    directory='kt_tuner_results',  # saves results here
    project_name='geopolitical',
    overwrite=True
)

print("✅ Keras Tuner ready!")
tuner.search_space_summary()

In [ ]:
# ── Run the search ─────────────────────────────────────────────────────────
print("🔍 Starting hyperparameter search...\n")
tuner.search(
    X_train_scaled, y_train_np,
    validation_data=(X_val_scaled, y_val_np),
    epochs=30,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
    verbose=1
)

print("\n✅ Hyperparameter search complete!")

In [ ]:
# ── Get the best hyperparameters ──────────────────────────────────────────
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("🏆 Best Hyperparameters Found:")
print(f"   Number of layers : {best_hps.get('num_layers')}")
for i in range(best_hps.get('num_layers')):
    print(f"   Layer {i+1} units   : {best_hps.get(f'units_{i}')}")
    print(f"   Layer {i+1} dropout : {best_hps.get(f'dropout_{i}')}")
print(f"   Learning rate    : {best_hps.get('learning_rate')}")

In [ ]:
# ── Build and train the best model ────────────────────────────────────────
best_model = tuner.hypermodel.build(best_hps)
best_model.summary()

print("\n🚀 Training best model with optimal hyperparameters...\n")
best_history = best_model.fit(
    X_train_scaled, y_train_np,
    validation_data=(X_val_scaled, y_val_np),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
    ],
    verbose=1
)

In [ ]:
# ── Final evaluation of best model ───────────────────────────────────────
_, best_test_acc = best_model.evaluate(X_test_scaled, y_test_np, verbose=0)
_, base_test_acc = model.evaluate(X_test_scaled, y_test_np, verbose=0)

print("🏁 FINAL COMPARISON:")
print(f"   Baseline model accuracy (Step 6) : {base_test_acc*100:.2f}%")
print(f"   Best tuned model accuracy        : {best_test_acc*100:.2f}%")
improvement = (best_test_acc - base_test_acc) * 100
print(f"   Improvement                      : {improvement:+.2f}%")

In [ ]:
# ── Final Confusion Matrix for best model ────────────────────────────────
y_best_pred = np.argmax(best_model.predict(X_test_scaled), axis=1)

plt.figure(figsize=(8, 6))
cm_best = confusion_matrix(y_test_np, y_best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=class_names)
disp.plot(cmap='Greens', ax=plt.gca(), colorbar=False)
plt.title('Confusion Matrix — Best Tuned Model (Test Set)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("\n📋 Final Classification Report:")
print(classification_report(y_test_np, y_best_pred,
                             target_names=class_names, zero_division=0))

---
## 💾 STEP 9 (Bonus) — Save the Best Model

In [ ]:
# Save the best model to disk
best_model.save('best_geopolitical_model.keras')
print("✅ Best model saved as 'best_geopolitical_model.keras'")

# To reload it later:
# loaded_model = keras.models.load_model('best_geopolitical_model.keras')

# Save the scaler too (use pickle)
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ StandardScaler saved as 'scaler.pkl'")
print("\n🎉 Project complete! Well done!")